In [1]:
# Kill all processes on GPU
!fuser -v /dev/nvidia* -k

In [2]:
%%capture
import os, importlib.util
%pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

# Configuration

In [3]:
# Fix TorchCodec issue
import os
os.environ["DISABLE_TORCHCODEC"] = "1"

In [4]:
SEED = 42
LANG = 'en'
TASK = 'read'

# Model configuration
MODEL_ID = "unsloth/Qwen3.5-2B"
RESUME_MODEL_ID = None

# Data configuration
DATA_SIZE = 1250
TEST_RATIO = 0.1

# Training configuration
MAX_SEQ_LENGTH = 2048

In [5]:
from datetime import datetime

# Resume training configuration
resume_from_checkpoint = bool(RESUME_MODEL_ID)
if resume_from_checkpoint:
    save_model_id = RESUME_MODEL_ID
    project_name = save_model_id.split('/')[-1]
    model_name = project_name

    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=save_model_id, local_dir=model_name)
else:
    # model_name = 'unsloth/Meta-Llama-3.1-8B'
    user_name, model_name = MODEL_ID.split('/')
    project_name = f'{model_name}-{TASK}-{LANG}-LoRA-v{datetime.now().strftime("%Y%m%d%H%M%S")}'
    save_model_id = f'alxxtexxr/{project_name}'
print("Resume from checkpoint:", resume_from_checkpoint)
print("Project name:", project_name)
print("Save model ID:", save_model_id)

Resume from checkpoint: False
Project name: Qwen3.5-2B-read-en-LoRA-v20260418191612
Save model ID: alxxtexxr/Qwen3.5-2B-read-en-LoRA-v20260418191612


# Model

In [6]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

In [7]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False, since we don't work with images
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = SEED,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

# Data

In [8]:
from datasets import load_dataset, Dataset, DatasetDict

def load_hf_dataset(lang, task, split, size=1000, test_ratio=0.1, seed=42):
    # Map task to dataset
    data_id_map = {
        'read': 'wikimedia/wikipedia',
        'math': 'openai/gsm8k',
    }
    data_id = data_id_map[task]
    data_dir = f'20231101.{lang}' if task == 'read' else 'main'

    # Streaming load
    dataset_stream = load_dataset(data_id, data_dir=data_dir, split=split, streaming=True)

    # Take `size` samples
    sliced_data = []
    for i, example in enumerate(dataset_stream):
        if i >= size:
            break
        sliced_data.append(example)

    # Convert to Dataset
    dataset = Dataset.from_list(sliced_data)

    # Shuffle
    dataset = dataset.shuffle(seed=seed)

    # First split: train vs temp (val+test)
    split_dict = dataset.train_test_split(
        test_size=test_ratio*2,
        seed=seed
    )

    train_dataset = split_dict["train"]
    temp_dataset = split_dict["test"]

    # Second split: val vs test
    temp_split = temp_dataset.train_test_split(
        test_size=0.5,
        seed=seed
    )

    val_dataset = temp_split["train"]
    test_dataset = temp_split["test"]

    # Return DatasetDict
    return DatasetDict({
        "train": train_dataset,
        "validation": val_dataset,
        "test": test_dataset,
    })

dataset = load_hf_dataset(
    lang=LANG, 
    task=TASK, 
    split='train',
    size=DATA_SIZE, 
    test_ratio=TEST_RATIO, 
    seed=SEED,
)
dataset

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 125
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 125
    })
})

In [9]:
train_dataset = dataset["train"]
val_dataset = dataset["validation"]
# test_dataset = dataset["test"]

train_dataset[0]['text'][-100:]

"merican Standard Version, Young's Literal Translation)\n\n \n8th-century BC books\nTwelve Minor Prophets"

In [10]:
def tokenize(example):
    return tokenizer(
        None, # Image is None for LLMs, but required for vision fine-tuning
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

train_dataset_tokenized = train_dataset.map(tokenize, batched=True, remove_columns=train_dataset.column_names, num_proc=8)
val_dataset_tokenized = val_dataset.map(tokenize, batched=True, remove_columns=val_dataset.column_names, num_proc=8)
# test_dataset_tokenized = test_dataset.map(tokenize, batched=True, remove_columns=test_dataset.column_names, num_proc=8)

Map (num_proc=8):   0%|          | 0/1000 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/125 [00:00<?, ? examples/s]

# Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth import UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported


trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset_tokenized,
    eval_dataset = val_dataset_tokenized,
    # dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 8,

    args = UnslothTrainingArguments(
        # Training arguments
        seed = SEED,
        
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,

        num_train_epochs = 10,
        # warmup_ratio = 0.05,
        warmup_steps = 10, # Supposeing 5% of total 200 steps
        learning_rate = 1e-4,
        # embedding_learning_rate = 5e-6,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        max_grad_norm = 1.0,
        weight_decay = 0.01,
        
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        tf32 = False,

        # Validation arguments
        eval_strategy='steps',
        eval_steps=20,
        
        # Logging arguments
        logging_strategy='steps',
        logging_steps=10,
        # logging_first_step=True,
        report_to=['tensorboard', 'wandb'],
        
        # Saving arguments
        save_strategy='steps',
        # save_steps=1, # For testing
        save_steps=20,
        # save_total_limit=5, # 1 best + 4 recent checkpoints. Warning: It doesn't work
        
        # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
        # So you will find one checkpoint at the end of each epoch.
        # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
        load_best_model_at_end=True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,

        output_dir=project_name,
        hub_model_id=save_model_id,
        push_to_hub=True,
        hub_strategy='all_checkpoints',
        hub_always_push=True,
        
        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        # dataset_text_field = "text",
        dataset_kwargs = {},
        max_length = MAX_SEQ_LENGTH,
    ),
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience = 2,
            early_stopping_threshold = 0.001,
        )
    ],
)

Unsloth: Switching to float32 training since model cannot work with float16


In [12]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 10 | Total steps = 630
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 16,819,200 of 2,230,060,864 (0.75% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.81 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.93 GiB is free. Including non-PyTorch memory, this process has 12.63 GiB memory in use. Of the allocated memory 12.13 GiB is allocated by PyTorch, and 216.67 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)